In [ ]:
!pip install pycirclize

In [ ]:
from pycirclize import Circos
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# Load contacts file
df_links = pd.read_csv("Protein_structure_prediction/TWN_Interaction/Dimer/Full_Data/contacts_interchain_merged_thr0.5.csv")

# Chain -> Protein mapping
chain_to_protein = {
    "t1": "TSC22D1","t2": "TSC22D2","t3": "TSC22D3","t4": "TSC22D4",
    "w1": "WNK1","w2": "WNK2","w3": "WNK3","w4": "WNK4",
    "n1": "NRBP1","n2": "NRBP2",
    "s": "STK39","o": "OXSR1",
}

# Define sectors
sectors = {
    "TSC22D1": 1073, "TSC22D2": 780, "TSC22D3": 134, "TSC22D4": 395,
    "WNK1": 2382, "WNK2": 2297, "WNK3": 1800, "WNK4": 1243,
    "NRBP1": 535, "NRBP2": 501, "STK39": 545, "OXSR1": 527
}

# Protein colors
protein_colors = {
    "TSC22D1": "#ffe5b6","TSC22D2": "#ffe5b6","TSC22D3": "#ffe5b6","TSC22D4": "#ffe5b6",
    "WNK1": "#49c1bb","WNK2": "#49c1bb","WNK3": "#49c1bb","WNK4": "#49c1bb",
    "NRBP1": "#bababa","NRBP2": "#bababa",
    "STK39": "#6dabc6","OXSR1": "#6dabc6",
}

# Color mixing function
def mix_colors(hex1, hex2):
    rgb1 = np.array(mcolors.to_rgb(hex1))
    rgb2 = np.array(mcolors.to_rgb(hex2))
    mixed = (rgb1 + rgb2) / 2
    return mcolors.to_hex(mixed)

# Circos base
circos = Circos(sectors, space=2)

# Add protein color track
for sector in circos.sectors:
    protein = sector.name
    track = sector.add_track((90, 100))
    track.axis(ec="none", fc="none")
    values = np.ones(sectors[protein])
    color = protein_colors.get(protein, "#cccccc")
    track.heatmap(values[np.newaxis, :], cmap=mcolors.ListedColormap([color]),
                  vmin=0, vmax=1, rect_kws=dict(ec="none"))

# Add links with mixed colors
for _, row in df_links.iterrows():
    res1, chain1 = row["Residue 1"], row["Chain 1"]
    res2, chain2 = row["Residue 2"], row["Chain 2"]
    prob = row["Probability"]

    if chain1 not in chain_to_protein or chain2 not in chain_to_protein:
        continue
    prot1, prot2 = chain_to_protein[chain1], chain_to_protein[chain2]

    alpha = min(max(prob, 0.5), 1.0)

    color1 = protein_colors.get(prot1, "#b3b3b3")
    color2 = protein_colors.get(prot2, "#b3b3b3")
    line_color = color1 if color1 == color2 else mix_colors(color1, color2)

    circos.link(
        (prot1, int(res1), int(res1)),
        (prot2, int(res2), int(res2)),
        r1=85, r2=85,
        color=line_color,
        lw=0.1,
        alpha=alpha
    )

# Save
fig = circos.plotfig()
fig.savefig("Figure_5A_Binary_Interaction_mixed.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("Figure_5A_Binary_Interaction_mixed.png", dpi=800, bbox_inches="tight", transparent=True)
plt.show()
